In [1]:
import os, re, torch
from pathlib import Path
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'USING DEVICE: {device}')

# InLegalBERT for query side
legal_embeddings = HuggingFaceEmbeddings(
    model_name = 'law-ai/InLegalBERT',
    model_kwargs = {'device': device},
    encode_kwargs = {'normalize_embeddings': True}
)

# BGE-Base for passage side encoding(broader coverage)

passage_embeddings = HuggingFaceEmbeddings(
    model_name = 'BAAI/bge-base-en-v1.5',
    model_kwargs = {'device':device},
    encode_kwargs={
        'normalize_embeddings': True,
        # BGE models look for instructions inside the 'prompt' field under encode_kwargs
        'prompt': "Represent this sentence for searching relevant passages: "
    }
)

# sentence model for semantic chunking similarity calculations

sentence_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

DATA_DIR = Path('../data/raw')
DB_DIR = Path('../data/vector_database')
DB_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(DB_DIR))
print(f'ChromeDB client initialized at {DB_DIR}')


python-dotenv could not parse statement starting at line 51
python-dotenv could not parse statement starting at line 52
python-dotenv could not parse statement starting at line 53
No sentence-transformers model found with name law-ai/InLegalBERT. Creating a new one with mean pooling.


USING DEVICE: cuda


InvalidURL: Failed to parse: http://your-proxy:port

## Document Cleaning

In [ ]:
import unicodedata 
from langchain_community.document_loaders import PyMuPDFLoader

def clean_legal_text(text: str) -> str:

    """ 
    Production-grade legal text cleaner. 
    Handles: Non-ASCII noise, page footers, PDF artifacts, 
    iLovePDF stamps, watermarks, header repetition. 
    """
    # Remove non_ASCII (Hindi unicode that slipped through OCR)
    text = ''.join(ch for ch in text if ord(ch)<128)

    # remove footer separator lines
    text = re.split(r'_{2,}', text)[0]

    # Remove common PDF watermarks 
    text = re.sub(r'iLovePDF|www\.ilove.*?com', '', text, flags=re.IGNORECASE) 
    
    # Remove page number patterns like '\n42\n' 
    text = re.sub(r'\n\s*\d{1,4}\s*\n', '\n', text) 
    
    # Collapse multiple blank lines 
    text = re.sub(r'\n{3,}', '\n\n', text) 
    
    # Normalize whitespace 
    text = ' '.join(text.split()) 
    
    return text.strip()


def load_and_clean_pdf(path: str) -> list:
    loader = PyMuPDFLoader(path)
    pages = loader.load()
    for page in pages:
        page.page_content = clean_legal_text(page.page_content)
        page.metadata['source_file'] = Path(path).name
    
    # remove empty pages
    pages = [p for p in pages if len(p.page_content) >50]
    print(f' Loaded {len(pages)} non-empty pages from {Path(path).name}') 
    return pages

## Hierarchical Chunking Engine import

In [ ]:
import nltk, numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

nltk.download('punkt', quiet=True)

SIMILARITY_THRESHOLD = 0.78 
MAX_CHUNK_SIZE = 500 
CHUNK_OVERLAP = 100


splitter = RecursiveCharacterTextSplitter(
    chunk_size = MAX_CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', '']
)

def semantic_chunk(text: str)->list[str]:
    sentences = nltk.sent_tokenize(text)
    if len(sentences) <=2:
        return [text]
    
    prefixed = [f'Represent this sentence for retrieval: {s}' for s in sentences]
    embs = sentence_model.encode(prefixed, normalize_embeddings=True, show_progress_bar=False)

    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        sim = float(np.dot(embs[i], embs[i - 1]))
        if sim < SIMILARITY_THRESHOLD:
            chunks.append(''.join(current))
            current = []
        current.append(sentences[i])
    if current: 
        chunks.append(' '.join(current)) 
    return chunks


def hierarchical_chunk(pages: list) -> tuple[list, list]: 
    """
    Returns (parent_docs, child_docs) for parent-child retrieval.
    """ 
    parent_docs, child_docs = [], [] 
    
    for page in pages: 
        lines = page.page_content.split('\n') 
        reference = lines[0][:80].strip() if lines else 'General Provision' 
        parents = semantic_chunk(page.page_content) 
        for p_idx, parent_text in enumerate(parents): 
            p_meta = {**page.metadata, 'chunk_type': 'parent', 'parent_index': p_idx} 
            parent_docs.append(Document(page_content=parent_text, metadata=p_meta))
            for child_text in splitter.split_text(parent_text): 
                tagged = ( 
                    f'LAW_SECTION_REFERENCE: {reference}\n\n' 
                    f'CONTENT: {child_text}' 
                    ) 
                child_docs.append(Document( 
                    page_content=tagged, 
                    metadata={ 
                        **page.metadata, 
                        'chunk_type' : 'child', 
                        'reference' : reference, 
                        'parent_chunk' : parent_text[:200], 
                        } 
                        )) 
                
    return parent_docs, child_docs

## Batch Processing all Documents

In [ ]:
# Process all documents

all_child_docs = []

pdf_files = list(DATA_DIR.glob('*.pdf')) + list(DATA_DIR.glob('*.PDF'))
print(f'Found {len(pdf_files)} PDF files to process\n')

for pdf_path in pdf_files:
    print(f'Processing: {pdf_path.name}')
    pages = load_and_clean_pdf(str(pdf_path))
    parents, children = hierarchical_chunk(pages)
    all_child_docs.extend(children)
    print(f' -> {len(children)} child chunks created\n')

print(f'Total child chunks ready for embedding: {len(all_child_docs)}')

Found 12 PDF files to process

Processing: constitution.pdf
 Loaded 401 non-empty pages from constitution.pdf


C:\Users\user\AppData\Roaming\Python\Python311\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


 -> 5177 child chunks created

Processing: contract_act.pdf
 Loaded 52 non-empty pages from contract_act.pdf
 -> 1939 child chunks created

Processing: cpc_1908.pdf
 Loaded 309 non-empty pages from cpc_1908.pdf
 -> 7903 child chunks created

Processing: crpc_1973.pdf
 Loaded 30 non-empty pages from crpc_1973.pdf
 -> 186 child chunks created

Processing: evidence_act.pdf
 Loaded 60 non-empty pages from evidence_act.pdf
 -> 1701 child chunks created

Processing: ipc_1860.pdf
 Loaded 28 non-empty pages from ipc_1860.pdf
 -> 537 child chunks created

Processing: constitution.pdf
 Loaded 401 non-empty pages from constitution.pdf
 -> 5177 child chunks created

Processing: contract_act.pdf
 Loaded 52 non-empty pages from contract_act.pdf
 -> 1939 child chunks created

Processing: cpc_1908.pdf
 Loaded 309 non-empty pages from cpc_1908.pdf
 -> 7903 child chunks created

Processing: crpc_1973.pdf
 Loaded 30 non-empty pages from crpc_1973.pdf
 -> 186 child chunks created

Processing: evidence_act

## Vector Store Creation & Verification

In [ ]:
# build vector store

COLLECTION = 'legal_knowledge_v2'

# purge old collection to prevent vector pollution

try:
    client.delete_collection(COLLECTION)
    print(f'Purged old collection: {COLLECTION}')
except Exception as e:
    print(f'No existing collection: {e}')

# build new store using BGE-Base for passage encoding

vector_store = Chroma.from_documents(
    documents= all_child_docs,
    embedding= passage_embeddings, # bge-base for passages
    client= client,
    collection_name=COLLECTION,
)

count = vector_store._collection.count()
print(f'Vector store built: {count} chunks')

# verifcation query using InLegalBERT

test = 'Right against self-incrimination Article 20' 
results = vector_store.similarity_search(test, k=3) 
print(f'\n🔍 Test retrieval for: "{test}"') 
for i, r in enumerate(results, 1): 
    print(f'Result {i}: {r.page_content[:200]}')

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


No existing collection: Collection legal_knowledge_v2 does not exist.


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Vector store built: 34886 chunks

🔍 Test retrieval for: "Right against self-incrimination Article 20"
Result 1: LAW_SECTION_REFERENCE: 293 against him; or to appear and show cause why he should not furnish security;

CONTENT: GIVEN under my hand and the seal of the Court, this..day of20.
Result 2: LAW_SECTION_REFERENCE: 293 against him; or to appear and show cause why he should not furnish security;

CONTENT: GIVEN under my hand and the seal of the Court, this..day of20.
Result 3: LAW_SECTION_REFERENCE: 300 WHEREAS you were a party in Suit No.of20, in the Court of,., and whereas the

CONTENT: GIVEN under my hand and the seal of the Court, this..day of.20.. Judge.
